In [ ]:
!pip install dask[complete] dask-ml pandas scikit-learn --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.0/150.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.4/259.4 kB 18.6 MB/s eta 0:00:00


In [ ]:
import time

import dask.dataframe as dd
import dask.array as da
from dask_ml.model_selection import train_test_split as dask_train_test_split
from dask_ml.linear_model import LogisticRegression as DaskLogReg

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

In [ ]:
import dask.dataframe as dd

ddf = dd.read_csv('/diabetes.csv')

In [ ]:
ddf = ddf.dropna()

In [ ]:
numeric_features = [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction",
    "Age"
]

X_dd = ddf[numeric_features].astype("float32")
y_dd = ddf["Outcome"].astype("int8")

In [ ]:

start = time.perf_counter()

df = ddf.compute()
n_rows = df.shape[0]

X = df[numeric_features].astype("float32")
y = df["Outcome"].astype("int8")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf_sklearn = LogisticRegression(max_iter=500)
clf_sklearn.fit(X_train, y_train)
acc_sklearn = clf_sklearn.score(X_test, y_test)

end = time.perf_counter()
time_sklearn = end - start

print("[Baseline] sklearn LogisticRegression")
print(f"  Accuracy: {acc_sklearn:.4f}")
print(f"  Time: {time_sklearn:.3f} seconds")

[Baseline] sklearn LogisticRegression
  Accuracy: 0.7143
  Time: 0.110 seconds


In [ ]:
start = time.perf_counter()
X_da = X_dd.to_dask_array(lengths=True)
y_da = y_dd.to_dask_array(lengths=True)

X_train_da, X_test_da, y_train_da, y_test_da = dask_train_test_split(
    X_da, y_da, test_size=0.2, random_state=42
)

clf_dask = DaskLogReg(max_iter=500)
clf_dask.fit(X_train_da, y_train_da)

acc_dask = clf_dask.score(X_test_da, y_test_da).compute()

end = time.perf_counter()
time_dask = end - start

print("[Dask-ML] Distributed LogisticRegression")
print(f"  Accuracy: {acc_dask:.4f}")
print(f"  Time: {time_dask:.3f} seconds")



[Dask-ML] Distributed LogisticRegression
  Accuracy: 0.7532
  Time: 0.970 seconds


In [ ]:
print("\n=== Summary ===")
print(f"sklearn  accuracy: {acc_sklearn:.4f}, time: {time_sklearn:.3f} s")
print(f"Dask-ML accuracy: {acc_dask:.4f}, time: {time_dask:.3f} s")


=== Summary ===
sklearn  accuracy: 0.7143, time: 0.110 s
Dask-ML accuracy: 0.7532, time: 0.970 s
